# Advanced Problems with Solutions: Coroutines, `deque`, and Cooperative Scheduling

This notebook develops advanced, runnable exercises around:

- `collections.deque` as a FIFO queue, bounded buffer, and round-robin ready queue
- generator-based coroutines using `yield`, `send`, `throw`, and `close`
- cooperative producer-consumer systems
- backpressure, fairness, cancellation, cleanup, and fault isolation
- timed scheduling and a small event loop
- correct microbenchmark design
- a capstone streaming ETL system

> **Modern Python note:** Generator-based coroutines are excellent for understanding cooperative execution. For production network I/O, prefer `async def`, `await`, and `asyncio`.

Every problem contains a specification, complete solution, assertions, and design notes. Only the Python standard library is used.


## Learning objectives

By the end, you should be able to:

1. choose the correct end of a `deque` for FIFO behavior;
2. distinguish an iterator generator from a coroutine-style generator;
3. prime and drive coroutines safely;
4. use `send`, `throw`, and `close`;
5. implement explicit bounded-queue overflow policies;
6. build fair round-robin and timed schedulers;
7. model backpressure, retries, cancellation, and cleanup;
8. write benchmarks that do not mutate shared fixtures.


In [1]:
from __future__ import annotations

from collections import Counter, deque
from collections.abc import Callable, Generator, Iterable, Iterator
from dataclasses import dataclass, field
from enum import Enum, auto
from functools import wraps
import heapq
import inspect
import itertools
import math
import statistics
import timeit
from typing import Any, Generic, TypeVar

T = TypeVar("T")
U = TypeVar("U")


## Coroutine helpers

A generator must be suspended at its first `yield` before receiving a non-`None` value through `.send(...)`.


In [2]:
def prime(coroutine_object: Generator[None, T, U]) -> Generator[None, T, U]:
    """Advance a coroutine to its first yield and return it."""
    next(coroutine_object)
    return coroutine_object


def coroutine(
    function: Callable[..., Generator[None, T, U]],
) -> Callable[..., Generator[None, T, U]]:
    """Decorator that automatically primes a generator-based coroutine."""
    @wraps(function)
    def wrapper(*args: Any, **kwargs: Any) -> Generator[None, T, U]:
        return prime(function(*args, **kwargs))

    return wrapper


## Example 0A — FIFO orientation with `deque`

Two equivalent FIFO conventions are common:

- enqueue with `append`, dequeue with `popleft`;
- enqueue with `appendleft`, dequeue with `pop`.

This notebook normally uses `append` + `popleft`.


In [3]:
queue = deque[int]()

for value in [10, 20, 30]:
    queue.append(value)

removed = [queue.popleft(), queue.popleft(), queue.popleft()]

assert removed == [10, 20, 30]
assert not queue
removed


[10, 20, 30]

## Example 0B — Receiving values with `.send(...)`

Here, `yield` is a suspension point rather than an iterator output.


In [4]:
@coroutine
def running_total() -> Generator[None, float, None]:
    total = 0.0

    while True:
        value = yield
        total += value
        print(f"received={value:>5}, total={total:>6.1f}")


totaler = running_total()
totaler.send(2.5)
totaler.send(4.0)
totaler.send(-1.5)
totaler.close()


received=  2.5, total=   2.5
received=  4.0, total=   6.5
received= -1.5, total=   5.0


# Problem 1 — Bounded FIFO with explicit overflow policies

Implement a generic queue that:

- stores items in FIFO order;
- has a positive fixed capacity;
- either rejects a new item or explicitly evicts the oldest item;
- tracks accepted, rejected, and evicted counts;
- raises `IndexError` when read while empty.

For work queues, explicit data-loss behavior is safer than silent eviction by `deque(maxlen=...)`.


In [5]:
class OverflowPolicy(Enum):
    REJECT_NEW = auto()
    DROP_OLDEST = auto()


@dataclass(frozen=True)
class PutResult(Generic[T]):
    accepted: bool
    evicted: T | None = None


class BoundedFIFO(Generic[T]):
    """A bounded FIFO queue with observable overflow behavior."""

    def __init__(
        self,
        capacity: int,
        policy: OverflowPolicy = OverflowPolicy.REJECT_NEW,
    ) -> None:
        if capacity <= 0:
            raise ValueError("capacity must be positive")

        self._capacity = capacity
        self._policy = policy
        self._items: deque[T] = deque()

        self.accepted_count = 0
        self.rejected_count = 0
        self.evicted_count = 0

    @property
    def capacity(self) -> int:
        return self._capacity

    @property
    def policy(self) -> OverflowPolicy:
        return self._policy

    def __len__(self) -> int:
        return len(self._items)

    def __bool__(self) -> bool:
        return bool(self._items)

    def is_full(self) -> bool:
        return len(self) == self.capacity

    def put(self, item: T) -> PutResult[T]:
        if not self.is_full():
            self._items.append(item)
            self.accepted_count += 1
            return PutResult(accepted=True)

        if self.policy is OverflowPolicy.REJECT_NEW:
            self.rejected_count += 1
            return PutResult(accepted=False)

        evicted = self._items.popleft()
        self._items.append(item)
        self.accepted_count += 1
        self.evicted_count += 1
        return PutResult(accepted=True, evicted=evicted)

    def get(self) -> T:
        if not self._items:
            raise IndexError("get from empty BoundedFIFO")
        return self._items.popleft()

    def peek(self) -> T:
        if not self._items:
            raise IndexError("peek from empty BoundedFIFO")
        return self._items[0]

    def snapshot(self) -> tuple[T, ...]:
        return tuple(self._items)


In [6]:
# Reject-new policy
rejecting = BoundedFIFO[int](3, OverflowPolicy.REJECT_NEW)

assert rejecting.put(1) == PutResult(True)
assert rejecting.put(2) == PutResult(True)
assert rejecting.put(3) == PutResult(True)
assert rejecting.put(4) == PutResult(False)
assert rejecting.snapshot() == (1, 2, 3)
assert rejecting.accepted_count == 3
assert rejecting.rejected_count == 1
assert [rejecting.get(), rejecting.get(), rejecting.get()] == [1, 2, 3]

# Drop-oldest policy
rolling = BoundedFIFO[str](3, OverflowPolicy.DROP_OLDEST)

for item in ["A", "B", "C"]:
    assert rolling.put(item).accepted

result = rolling.put("D")

assert result == PutResult(accepted=True, evicted="A")
assert rolling.snapshot() == ("B", "C", "D")
assert rolling.evicted_count == 1


### Complexity

- `put`, `get`, and `peek`: **O(1)**
- `snapshot`: **O(n)** because it copies all items
- equivalent left-end list operations: **O(n)**


# Problem 2 — Fair round-robin scheduler

A task is a generator. Each `next(task)` performs one cooperative step.

Build a scheduler that:

- stores ready tasks in a `deque`;
- runs one step per task per turn;
- requeues unfinished tasks;
- records completion values;
- isolates task failures;
- proves deterministic fairness.


In [7]:
@dataclass
class TaskFailure:
    task_name: str
    error: BaseException


@dataclass
class RoundRobinReport:
    completions: dict[str, Any] = field(default_factory=dict)
    failures: list[TaskFailure] = field(default_factory=list)
    execution_trace: list[str] = field(default_factory=list)


@dataclass
class NamedTask:
    name: str
    generator: Generator[str, None, Any]


class RoundRobinScheduler:
    def __init__(self) -> None:
        self._ready: deque[NamedTask] = deque()
        self.report = RoundRobinReport()

    def add(self, name: str, generator: Generator[str, None, Any]) -> None:
        if any(task.name == name for task in self._ready):
            raise ValueError(f"duplicate task name: {name}")
        self._ready.append(NamedTask(name, generator))

    def run(self, *, max_steps: int | None = None) -> RoundRobinReport:
        steps = 0

        while self._ready:
            if max_steps is not None and steps >= max_steps:
                break

            task = self._ready.popleft()

            try:
                event = next(task.generator)
            except StopIteration as stop:
                self.report.completions[task.name] = stop.value
            except BaseException as error:
                self.report.failures.append(TaskFailure(task.name, error))
            else:
                self.report.execution_trace.append(event)
                self._ready.append(task)

            steps += 1

        return self.report


In [8]:
def counting_task(name: str, steps: int) -> Generator[str, None, str]:
    for index in range(1, steps + 1):
        yield f"{name}:{index}"
    return f"{name} completed"


def failing_task() -> Generator[str, None, None]:
    yield "FAIL:before-error"
    raise RuntimeError("simulated failure")


scheduler = RoundRobinScheduler()
scheduler.add("A", counting_task("A", 3))
scheduler.add("B", counting_task("B", 2))
scheduler.add("C", counting_task("C", 4))
scheduler.add("FAIL", failing_task())

report = scheduler.run()

assert report.execution_trace == [
    "A:1",
    "B:1",
    "C:1",
    "FAIL:before-error",
    "A:2",
    "B:2",
    "C:2",
    "A:3",
    "C:3",
    "C:4",
]
assert report.completions["A"] == "A completed"
assert report.completions["B"] == "B completed"
assert report.completions["C"] == "C completed"
assert len(report.failures) == 1
assert report.failures[0].task_name == "FAIL"

report


RoundRobinReport(completions={'B': 'B completed', 'A': 'A completed', 'C': 'C completed'}, failures=[TaskFailure(task_name='FAIL', error=RuntimeError('simulated failure'))], execution_trace=['A:1', 'B:1', 'C:1', 'FAIL:before-error', 'A:2', 'B:2', 'C:2', 'A:3', 'C:3', 'C:4'])

A cooperative scheduler is fair only among tasks that yield promptly. A task that performs long CPU work before yielding blocks every other task.


# Problem 3 — Producer-consumer backpressure

Simulate a bounded producer-consumer system where:

- the producer pauses while the queue is full;
- the consumer performs bounded work per turn;
- no item is overwritten;
- the consumed sequence exactly matches the source.


In [9]:
@dataclass
class FlowMetrics:
    producer_steps: int = 0
    consumer_steps: int = 0
    producer_blocked: int = 0
    max_queue_size: int = 0


def producer_task(
    values: Iterable[int],
    queue: BoundedFIFO[int],
    metrics: FlowMetrics,
) -> Generator[str, None, None]:
    for value in values:
        while queue.is_full():
            metrics.producer_blocked += 1
            yield "producer:blocked"

        assert queue.put(value).accepted
        metrics.producer_steps += 1
        metrics.max_queue_size = max(metrics.max_queue_size, len(queue))
        yield f"producer:put:{value}"


def consumer_task(
    queue: BoundedFIFO[int],
    output: list[int],
    metrics: FlowMetrics,
    *,
    producer_done: Callable[[], bool],
    batch_size: int,
) -> Generator[str, None, None]:
    if batch_size <= 0:
        raise ValueError("batch_size must be positive")

    while not producer_done() or queue:
        processed_this_turn = 0

        while queue and processed_this_turn < batch_size:
            output.append(queue.get())
            metrics.consumer_steps += 1
            processed_this_turn += 1

        if processed_this_turn:
            yield f"consumer:processed:{processed_this_turn}"
        else:
            yield "consumer:idle"


In [10]:
source_values = list(range(1, 31))
buffer = BoundedFIFO[int](capacity=5)
processed: list[int] = []
flow_metrics = FlowMetrics()
flow_state = {"producer_done": False}


def wrapped_producer() -> Generator[str, None, None]:
    yield from producer_task(source_values, buffer, flow_metrics)
    flow_state["producer_done"] = True


flow_scheduler = RoundRobinScheduler()
flow_scheduler.add("producer", wrapped_producer())
flow_scheduler.add(
    "consumer",
    consumer_task(
        buffer,
        processed,
        flow_metrics,
        producer_done=lambda: flow_state["producer_done"],
        batch_size=2,
    ),
)

flow_report = flow_scheduler.run()

assert processed == source_values
assert len(buffer) == 0
assert flow_metrics.max_queue_size <= buffer.capacity
assert not flow_report.failures

flow_metrics


FlowMetrics(producer_steps=30, consumer_steps=30, producer_blocked=0, max_queue_size=1)

# Problem 4 — Online statistics with `.send(...)`

Build a coroutine that incrementally computes count, mean, population variance, minimum, and maximum using Welford's algorithm.

Commands:

- send a number to update;
- send `SNAPSHOT` to receive a snapshot;
- send `STOP` to terminate and return the final snapshot.


In [11]:
@dataclass(frozen=True)
class StatsSnapshot:
    count: int
    mean: float
    population_variance: float
    minimum: float | None
    maximum: float | None


SNAPSHOT = object()
STOP = object()


def online_stats() -> Generator[StatsSnapshot | None, float | object, StatsSnapshot]:
    count = 0
    mean = 0.0
    m2 = 0.0
    minimum: float | None = None
    maximum: float | None = None

    message = yield None

    while True:
        if message is SNAPSHOT:
            variance = m2 / count if count else 0.0
            message = yield StatsSnapshot(
                count=count,
                mean=mean,
                population_variance=variance,
                minimum=minimum,
                maximum=maximum,
            )
            continue

        if message is STOP:
            variance = m2 / count if count else 0.0
            return StatsSnapshot(
                count=count,
                mean=mean,
                population_variance=variance,
                minimum=minimum,
                maximum=maximum,
            )

        value = float(message)
        count += 1

        delta = value - mean
        mean += delta / count
        delta2 = value - mean
        m2 += delta * delta2

        minimum = value if minimum is None else min(minimum, value)
        maximum = value if maximum is None else max(maximum, value)

        message = yield None


In [12]:
stats = online_stats()
next(stats)

for value in [2, 4, 4, 4, 5, 5, 7, 9]:
    assert stats.send(value) is None

snapshot = stats.send(SNAPSHOT)

assert snapshot is not None
assert snapshot.count == 8
assert math.isclose(snapshot.mean, 5.0)
assert math.isclose(snapshot.population_variance, 4.0)
assert snapshot.minimum == 2.0
assert snapshot.maximum == 9.0

try:
    stats.send(STOP)
except StopIteration as stop:
    final_snapshot = stop.value

assert final_snapshot == snapshot
snapshot


StatsSnapshot(count=8, mean=5.0, population_variance=4.0, minimum=2.0, maximum=9.0)

# Problem 5 — Composable coroutine pipeline

Create reusable `collect`, `map_stage`, `filter_stage`, and `broadcast` coroutines.

The pipeline squares inputs, keeps even squares, and broadcasts them to two sinks. Closing upstream must close downstream.


In [13]:
@coroutine
def collect(target: list[T]) -> Generator[None, T, None]:
    try:
        while True:
            item = yield
            target.append(item)
    finally:
        pass


@coroutine
def map_stage(
    function: Callable[[T], U],
    target: Generator[None, U, Any],
) -> Generator[None, T, None]:
    try:
        while True:
            item = yield
            target.send(function(item))
    finally:
        target.close()


@coroutine
def filter_stage(
    predicate: Callable[[T], bool],
    target: Generator[None, T, Any],
) -> Generator[None, T, None]:
    try:
        while True:
            item = yield
            if predicate(item):
                target.send(item)
    finally:
        target.close()


@coroutine
def broadcast(
    targets: Iterable[Generator[None, T, Any]],
) -> Generator[None, T, None]:
    active_targets = list(targets)

    try:
        while True:
            item = yield
            survivors: list[Generator[None, T, Any]] = []

            for target in active_targets:
                try:
                    target.send(item)
                except StopIteration:
                    continue
                else:
                    survivors.append(target)

            active_targets = survivors
    finally:
        for target in active_targets:
            target.close()


In [14]:
left_output: list[int] = []
right_output: list[int] = []

pipeline = map_stage(
    lambda value: value * value,
    filter_stage(
        lambda value: value % 2 == 0,
        broadcast([
            collect(left_output),
            collect(right_output),
        ]),
    ),
)

for value in range(1, 11):
    pipeline.send(value)

pipeline.close()

expected = [4, 16, 36, 64, 100]
assert left_output == expected
assert right_output == expected

left_output


[4, 16, 36, 64, 100]

Use `finally` for deterministic cleanup. Do not swallow `GeneratorExit`, because that prevents `.close()` from working correctly.


# Problem 6 — Sliding-window anomaly detector

Use `deque(maxlen=...)` for a rolling baseline. Here automatic eviction is intentional.

Requirements:

- warm up until the window is full;
- compare each new value with the current baseline;
- report values beyond a z-score threshold;
- do not insert detected anomalies into the baseline.


In [15]:
@dataclass(frozen=True)
class Anomaly:
    value: float
    mean: float
    standard_deviation: float
    z_score: float


@coroutine
def anomaly_detector(
    *,
    window_size: int,
    z_threshold: float,
    on_anomaly: Callable[[Anomaly], None],
) -> Generator[None, float, None]:
    if window_size < 2:
        raise ValueError("window_size must be at least 2")
    if z_threshold <= 0:
        raise ValueError("z_threshold must be positive")

    history: deque[float] = deque(maxlen=window_size)

    while True:
        value = float((yield))

        if len(history) < window_size:
            history.append(value)
            continue

        mean = statistics.fmean(history)
        standard_deviation = statistics.pstdev(history)

        if standard_deviation == 0:
            is_anomaly = value != mean
            z_score = math.inf if is_anomaly else 0.0
        else:
            z_score = abs(value - mean) / standard_deviation
            is_anomaly = z_score > z_threshold

        if is_anomaly:
            on_anomaly(
                Anomaly(
                    value=value,
                    mean=mean,
                    standard_deviation=standard_deviation,
                    z_score=z_score,
                )
            )
        else:
            history.append(value)


In [16]:
anomalies: list[Anomaly] = []
detector = anomaly_detector(
    window_size=5,
    z_threshold=3.0,
    on_anomaly=anomalies.append,
)

for observation in [10.0, 10.1, 9.9, 10.2, 9.8, 10.0, 50.0, 10.1]:
    detector.send(observation)

detector.close()

assert len(anomalies) == 1
assert anomalies[0].value == 50.0
assert anomalies[0].z_score > 3.0

anomalies[0]


Anomaly(value=50.0, mean=10.0, standard_deviation=0.141421356237309, z_score=282.84271247462)

# Problem 7 — Exception injection and cleanup

Use `.throw(...)` to inject recoverable and fatal errors into a coroutine. The coroutine must always execute cleanup.


In [17]:
class RecoverableInputError(Exception):
    pass


class FatalInputError(Exception):
    pass


@coroutine
def resilient_writer(
    output: list[str],
    lifecycle: list[str],
) -> Generator[None, str, None]:
    lifecycle.append("opened")

    try:
        while True:
            try:
                line = yield
            except RecoverableInputError as error:
                lifecycle.append(f"recovered:{error}")
                continue
            except FatalInputError as error:
                lifecycle.append(f"fatal:{error}")
                return
            else:
                output.append(line)
    finally:
        lifecycle.append("closed")


In [18]:
written: list[str] = []
lifecycle: list[str] = []

writer = resilient_writer(written, lifecycle)
writer.send("alpha")
writer.throw(RecoverableInputError("bad row"))
writer.send("beta")

try:
    writer.throw(FatalInputError("corrupt stream"))
except StopIteration:
    pass

assert written == ["alpha", "beta"]
assert lifecycle == [
    "opened",
    "recovered:bad row",
    "fatal:corrupt stream",
    "closed",
]

written, lifecycle


(['alpha', 'beta'],
 ['opened', 'recovered:bad row', 'fatal:corrupt stream', 'closed'])

# Problem 8 — Inspect generator state transitions

Prove the `GEN_CREATED`, `GEN_SUSPENDED`, `GEN_RUNNING`, and `GEN_CLOSED` states with `inspect.getgeneratorstate`.


In [19]:
state_observations: list[str] = []


def state_probe() -> Generator[None, None, str]:
    current = probe
    state_observations.append(inspect.getgeneratorstate(current))
    yield
    state_observations.append(inspect.getgeneratorstate(current))
    return "done"


probe = state_probe()

assert inspect.getgeneratorstate(probe) == inspect.GEN_CREATED

next(probe)
assert inspect.getgeneratorstate(probe) == inspect.GEN_SUSPENDED
assert state_observations == [inspect.GEN_RUNNING]

local_names = set(probe.gi_frame.f_locals)
assert "current" in local_names

try:
    next(probe)
except StopIteration as stop:
    assert stop.value == "done"

assert state_observations[-1] == inspect.GEN_RUNNING
assert inspect.getgeneratorstate(probe) == inspect.GEN_CLOSED

state_observations


['GEN_RUNNING', 'GEN_RUNNING']

# Problem 9 — Timed cooperative scheduler

Tasks yield:

- `YieldNow()` to rejoin the ready queue;
- `Sleep(ticks)` to wake at a future logical tick.

Use a `deque` for ready tasks and a min-heap for sleepers. Logical time keeps tests deterministic.


In [20]:
@dataclass(frozen=True)
class YieldNow:
    pass


@dataclass(frozen=True)
class Sleep:
    ticks: int

    def __post_init__(self) -> None:
        if self.ticks < 0:
            raise ValueError("ticks cannot be negative")


SchedulerCommand = YieldNow | Sleep


@dataclass
class TimedTask:
    task_id: int
    name: str
    generator: Generator[SchedulerCommand, None, Any]


@dataclass
class TimedEvent:
    tick: int
    task_name: str
    command: SchedulerCommand | str


class TimedScheduler:
    def __init__(self) -> None:
        self.now = 0
        self._next_task_id = itertools.count(1)
        self._sequence = itertools.count()
        self._ready: deque[TimedTask] = deque()
        self._sleeping: list[tuple[int, int, TimedTask]] = []

        self.events: list[TimedEvent] = []
        self.completions: dict[str, Any] = {}
        self.failures: dict[str, BaseException] = {}

    def create_task(
        self,
        name: str,
        generator: Generator[SchedulerCommand, None, Any],
    ) -> int:
        task_id = next(self._next_task_id)
        self._ready.append(TimedTask(task_id, name, generator))
        return task_id

    def _wake_due_tasks(self) -> None:
        while self._sleeping and self._sleeping[0][0] <= self.now:
            _, _, task = heapq.heappop(self._sleeping)
            self._ready.append(task)

    def run(self) -> None:
        while self._ready or self._sleeping:
            self._wake_due_tasks()

            if not self._ready:
                self.now = self._sleeping[0][0]
                self._wake_due_tasks()

            task = self._ready.popleft()

            try:
                command = next(task.generator)
            except StopIteration as stop:
                self.completions[task.name] = stop.value
                self.events.append(TimedEvent(self.now, task.name, "completed"))
                continue
            except BaseException as error:
                self.failures[task.name] = error
                self.events.append(
                    TimedEvent(
                        self.now,
                        task.name,
                        f"failed:{type(error).__name__}",
                    )
                )
                continue

            self.events.append(TimedEvent(self.now, task.name, command))

            if isinstance(command, YieldNow):
                self._ready.append(task)
            elif isinstance(command, Sleep):
                wake_tick = self.now + command.ticks
                heapq.heappush(
                    self._sleeping,
                    (wake_tick, next(self._sequence), task),
                )
            else:
                self.failures[task.name] = TypeError(
                    f"unsupported command: {command!r}"
                )

            self.now += 1


In [21]:
def periodic(
    name: str,
    periods: int,
    delay: int,
) -> Generator[SchedulerCommand, None, str]:
    for _ in range(periods):
        yield Sleep(delay)
        yield YieldNow()
    return f"{name}:done"


timed = TimedScheduler()
timed.create_task("fast", periodic("fast", 3, 1))
timed.create_task("slow", periodic("slow", 2, 4))
timed.run()

assert timed.completions == {
    "fast": "fast:done",
    "slow": "slow:done",
}
assert not timed.failures
assert any(
    event.task_name == "slow" and isinstance(event.command, Sleep)
    for event in timed.events
)

timed.events


[TimedEvent(tick=0, task_name='fast', command=Sleep(ticks=1)),
 TimedEvent(tick=1, task_name='slow', command=Sleep(ticks=4)),
 TimedEvent(tick=2, task_name='fast', command=YieldNow()),
 TimedEvent(tick=3, task_name='fast', command=Sleep(ticks=1)),
 TimedEvent(tick=4, task_name='fast', command=YieldNow()),
 TimedEvent(tick=5, task_name='fast', command=Sleep(ticks=1)),
 TimedEvent(tick=6, task_name='slow', command=YieldNow()),
 TimedEvent(tick=7, task_name='fast', command=YieldNow()),
 TimedEvent(tick=8, task_name='slow', command=Sleep(ticks=4)),
 TimedEvent(tick=9, task_name='fast', command='completed'),
 TimedEvent(tick=12, task_name='slow', command=YieldNow()),
 TimedEvent(tick=13, task_name='slow', command='completed')]

# Problem 10 — Cooperative cancellation

Cancellation can only be observed at a cooperative boundary. Wrap a generator so the next step injects `TaskCancelled` with `.throw(...)`.


In [22]:
class TaskCancelled(Exception):
    pass


@dataclass
class CancellableTask:
    generator: Generator[SchedulerCommand, None, Any]
    cancel_requested: bool = False

    def request_cancel(self) -> None:
        self.cancel_requested = True

    def step(self) -> SchedulerCommand:
        if self.cancel_requested:
            self.cancel_requested = False
            return self.generator.throw(TaskCancelled())
        return next(self.generator)

    def close(self) -> None:
        self.generator.close()


def cancellable_worker(
    lifecycle: list[str],
) -> Generator[SchedulerCommand, None, str]:
    lifecycle.append("started")

    try:
        while True:
            lifecycle.append("working")
            yield YieldNow()
    except TaskCancelled:
        lifecycle.append("cancel-observed")
        return "cancelled"
    finally:
        lifecycle.append("cleaned-up")


In [23]:
cancel_lifecycle: list[str] = []
worker = CancellableTask(cancellable_worker(cancel_lifecycle))

assert isinstance(worker.step(), YieldNow)
assert isinstance(worker.step(), YieldNow)

worker.request_cancel()

try:
    worker.step()
except StopIteration as stop:
    cancellation_result = stop.value

assert cancellation_result == "cancelled"
assert cancel_lifecycle == [
    "started",
    "working",
    "working",
    "cancel-observed",
    "cleaned-up",
]

cancel_lifecycle


['started', 'working', 'working', 'cancel-observed', 'cleaned-up']

# Problem 11 — Weighted producer fairness

Three producers have weights 1, 2, and 3. A weight controls how many enqueue attempts the producer receives per turn. Stop early when backpressure occurs.


In [24]:
@dataclass
class ProducerState:
    name: str
    pending: deque[str]
    weight: int
    accepted: int = 0
    blocked_turns: int = 0


def weighted_producer_step(
    producer: ProducerState,
    queue: BoundedFIFO[str],
) -> None:
    attempts = 0

    while producer.pending and attempts < producer.weight:
        if queue.is_full():
            producer.blocked_turns += 1
            break

        item = producer.pending.popleft()
        assert queue.put(item).accepted

        producer.accepted += 1
        attempts += 1


def bounded_consumer_step(
    queue: BoundedFIFO[str],
    consumed: list[str],
    *,
    batch_size: int,
) -> None:
    for _ in range(batch_size):
        if not queue:
            break
        consumed.append(queue.get())


In [25]:
producers = [
    ProducerState("A", deque(f"A{i}" for i in range(1, 7)), weight=1),
    ProducerState("B", deque(f"B{i}" for i in range(1, 7)), weight=2),
    ProducerState("C", deque(f"C{i}" for i in range(1, 7)), weight=3),
]

mailbox = BoundedFIFO[str](capacity=5)
consumed_items: list[str] = []

while any(producer.pending for producer in producers) or mailbox:
    for producer in producers:
        weighted_producer_step(producer, mailbox)
        bounded_consumer_step(mailbox, consumed_items, batch_size=2)

expected_items = {
    f"{name}{index}"
    for name in "ABC"
    for index in range(1, 7)
}

assert len(consumed_items) == 18
assert set(consumed_items) == expected_items
assert len(set(consumed_items)) == len(consumed_items)
assert sum(producer.accepted for producer in producers) == 18

Counter(item[0] for item in consumed_items), producers


(Counter({'A': 6, 'B': 6, 'C': 6}),
 [ProducerState(name='A', pending=deque([]), weight=1, accepted=6, blocked_turns=0),
  ProducerState(name='B', pending=deque([]), weight=2, accepted=6, blocked_turns=0),
  ProducerState(name='C', pending=deque([]), weight=3, accepted=6, blocked_turns=0)])

# Problem 12 — Correct `list` versus `deque` benchmarks

Every timed call must create a fresh mutable fixture. Reusing an emptied global collection makes later repetitions meaningless.


In [26]:
def list_append_right(n: int) -> None:
    values: list[int] = []
    for value in range(n):
        values.append(value)


def deque_append_right(n: int) -> None:
    values: deque[int] = deque()
    for value in range(n):
        values.append(value)


def list_insert_left(n: int) -> None:
    values: list[int] = []
    for value in range(n):
        values.insert(0, value)


def deque_append_left(n: int) -> None:
    values: deque[int] = deque()
    for value in range(n):
        values.appendleft(value)


def list_pop_right(n: int) -> None:
    values = list(range(n))
    while values:
        values.pop()


def deque_pop_right(n: int) -> None:
    values = deque(range(n))
    while values:
        values.pop()


def list_pop_left(n: int) -> None:
    values = list(range(n))
    while values:
        values.pop(0)


def deque_pop_left(n: int) -> None:
    values = deque(range(n))
    while values:
        values.popleft()


In [27]:
def benchmark_queue_operations(
    *,
    n: int = 2_000,
    repeat: int = 3,
) -> dict[str, float]:
    functions = {
        "list append right": list_append_right,
        "deque append right": deque_append_right,
        "list insert left": list_insert_left,
        "deque append left": deque_append_left,
        "list pop right": list_pop_right,
        "deque pop right": deque_pop_right,
        "list pop left": list_pop_left,
        "deque pop left": deque_pop_left,
    }

    results: dict[str, float] = {}

    for name, function in functions.items():
        timer = timeit.Timer(lambda fn=function: fn(n))
        results[name] = min(timer.repeat(repeat=repeat, number=1))

    return results


benchmark_results = benchmark_queue_operations()

for operation, seconds in sorted(
    benchmark_results.items(),
    key=lambda item: item[1],
):
    print(f"{operation:<20} {seconds:.6f} seconds")

assert benchmark_results["deque append left"] < benchmark_results["list insert left"]
assert benchmark_results["deque pop left"] < benchmark_results["list pop left"]


list append right    0.000069 seconds
deque append right   0.000073 seconds
deque append left    0.000078 seconds
list pop right       0.000097 seconds
deque pop left       0.000153 seconds
deque pop right      0.000218 seconds
list insert left     0.000843 seconds
list pop left        0.005875 seconds


# Problem 13 — Repair a broken coordinator

Common bugs in simple coroutine coordinators include:

- `range(1, n)` omits `n`;
- `deque(maxlen=...)` can silently discard work;
- the final partial batch may remain unconsumed;
- `finally` can obscure normal termination;
- producer completion is implicit.

The repaired version uses explicit source exhaustion, explicit backpressure, and a final drain.


In [28]:
def repaired_coordinator(
    n: int,
    *,
    capacity: int = 10,
    consumer_batch_size: int = 3,
) -> list[int]:
    if n < 0:
        raise ValueError("n cannot be negative")
    if capacity <= 0:
        raise ValueError("capacity must be positive")
    if consumer_batch_size <= 0:
        raise ValueError("consumer_batch_size must be positive")

    queue = BoundedFIFO[int](
        capacity=capacity,
        policy=OverflowPolicy.REJECT_NEW,
    )
    produced = iter(range(1, n + 1))
    consumed: list[int] = []
    source_exhausted = False

    while not source_exhausted or queue:
        while not queue.is_full() and not source_exhausted:
            try:
                item = next(produced)
            except StopIteration:
                source_exhausted = True
            else:
                assert queue.put(item).accepted

        for _ in range(consumer_batch_size):
            if not queue:
                break
            consumed.append(queue.get())

    return consumed


In [29]:
for n in [0, 1, 9, 10, 11, 35, 100]:
    result = repaired_coordinator(
        n,
        capacity=7,
        consumer_batch_size=4,
    )
    assert result == list(range(1, n + 1))

repaired_coordinator(20, capacity=6, consumer_batch_size=2)


[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

# Problem 14 — Capstone: cooperative streaming ETL with retries

Build a deterministic ETL pipeline:

1. producer enqueues raw records;
2. transformer validates and normalizes;
3. transient failures are retried;
4. permanent failures go to dead letters;
5. writer drains normalized records;
6. every stage performs bounded work;
7. no accepted record is silently lost;
8. shutdown requires all work queues to be empty.


In [30]:
class PermanentRecordError(ValueError):
    pass


class TransientRecordError(RuntimeError):
    pass


@dataclass
class RawEnvelope:
    record: dict[str, Any]
    attempt: int = 1


@dataclass
class ETLMetrics:
    produced: int = 0
    transformed: int = 0
    written: int = 0
    retries: int = 0
    dead_letters: int = 0
    blocked_input: int = 0
    blocked_output: int = 0
    turns: int = 0


@dataclass
class ETLState:
    source: deque[dict[str, Any]]
    input_queue: BoundedFIFO[RawEnvelope]
    output_queue: BoundedFIFO[dict[str, Any]]
    retry_queue: deque[RawEnvelope]
    written: list[dict[str, Any]]
    dead_letters: list[tuple[dict[str, Any], str]]
    transient_failures_remaining: dict[int, int]
    metrics: ETLMetrics = field(default_factory=ETLMetrics)


def normalize_record(
    record: dict[str, Any],
    transient_failures_remaining: dict[int, int],
) -> dict[str, Any]:
    if "id" not in record:
        raise PermanentRecordError("missing id")
    if "name" not in record:
        raise PermanentRecordError("missing name")

    record_id = record["id"]

    if not isinstance(record_id, int) or record_id <= 0:
        raise PermanentRecordError("id must be a positive integer")

    failures_left = transient_failures_remaining.get(record_id, 0)

    if failures_left > 0:
        transient_failures_remaining[record_id] = failures_left - 1
        raise TransientRecordError("simulated temporary dependency failure")

    name = str(record["name"]).strip()

    if not name:
        raise PermanentRecordError("name cannot be blank")

    return {
        "id": record_id,
        "name": name.title(),
    }


In [31]:
def producer_turn(state: ETLState, *, budget: int) -> bool:
    progressed = False

    for _ in range(budget):
        if not state.source:
            break
        if state.input_queue.is_full():
            state.metrics.blocked_input += 1
            break

        record = state.source.popleft()
        assert state.input_queue.put(RawEnvelope(record)).accepted

        state.metrics.produced += 1
        progressed = True

    return progressed


def transformer_turn(
    state: ETLState,
    *,
    budget: int,
    max_attempts: int,
    pending_normalized: deque[dict[str, Any]],
) -> bool:
    progressed = False

    for _ in range(budget):
        if pending_normalized:
            if state.output_queue.is_full():
                state.metrics.blocked_output += 1
                break

            normalized = pending_normalized.popleft()
            assert state.output_queue.put(normalized).accepted
            state.metrics.transformed += 1
            progressed = True
            continue

        if state.retry_queue:
            envelope = state.retry_queue.popleft()
        elif state.input_queue:
            envelope = state.input_queue.get()
        else:
            break

        try:
            normalized = normalize_record(
                envelope.record,
                state.transient_failures_remaining,
            )
        except PermanentRecordError as error:
            state.dead_letters.append((envelope.record, str(error)))
            state.metrics.dead_letters += 1
            progressed = True
            continue
        except TransientRecordError as error:
            if envelope.attempt < max_attempts:
                state.retry_queue.append(
                    RawEnvelope(
                        record=envelope.record,
                        attempt=envelope.attempt + 1,
                    )
                )
                state.metrics.retries += 1
            else:
                state.dead_letters.append(
                    (envelope.record, f"retry exhausted: {error}")
                )
                state.metrics.dead_letters += 1

            progressed = True
            continue

        if state.output_queue.is_full():
            pending_normalized.append(normalized)
            state.metrics.blocked_output += 1
            progressed = True
            break

        assert state.output_queue.put(normalized).accepted
        state.metrics.transformed += 1
        progressed = True

    return progressed


def writer_turn(state: ETLState, *, budget: int) -> bool:
    progressed = False

    for _ in range(budget):
        if not state.output_queue:
            break

        state.written.append(state.output_queue.get())
        state.metrics.written += 1
        progressed = True

    return progressed


In [32]:
def run_etl(
    records: Iterable[dict[str, Any]],
    *,
    input_capacity: int = 3,
    output_capacity: int = 2,
    producer_budget: int = 2,
    transformer_budget: int = 2,
    writer_budget: int = 1,
    max_attempts: int = 3,
    transient_failures: dict[int, int] | None = None,
) -> ETLState:
    state = ETLState(
        source=deque(records),
        input_queue=BoundedFIFO(input_capacity),
        output_queue=BoundedFIFO(output_capacity),
        retry_queue=deque(),
        written=[],
        dead_letters=[],
        transient_failures_remaining=dict(transient_failures or {}),
    )

    pending_normalized: deque[dict[str, Any]] = deque()

    while (
        state.source
        or state.input_queue
        or state.retry_queue
        or pending_normalized
        or state.output_queue
    ):
        state.metrics.turns += 1

        progress = False
        progress |= producer_turn(state, budget=producer_budget)
        progress |= transformer_turn(
            state,
            budget=transformer_budget,
            max_attempts=max_attempts,
            pending_normalized=pending_normalized,
        )
        progress |= writer_turn(state, budget=writer_budget)

        if not progress:
            raise RuntimeError("ETL deadlock: unfinished work but no progress")

    return state


In [33]:
records = [
    {"id": 1, "name": "  ada lovelace "},
    {"id": 2, "name": "grace hopper"},
    {"id": -3, "name": "invalid id"},
    {"name": "missing id"},
    {"id": 4, "name": "   "},
    {"id": 5, "name": "alan turing"},
    {"id": 6, "name": "katherine johnson"},
]

etl = run_etl(
    records,
    transient_failures={
        2: 1,
        5: 5,
    },
)

assert etl.written == [
    {"id": 1, "name": "Ada Lovelace"},
    {"id": 2, "name": "Grace Hopper"},
    {"id": 6, "name": "Katherine Johnson"},
]
assert len(etl.dead_letters) == 4
assert etl.metrics.produced == len(records)
assert etl.metrics.written == len(etl.written)
assert etl.metrics.retries == 3
assert etl.metrics.produced == etl.metrics.written + etl.metrics.dead_letters
assert etl.metrics.transformed == etl.metrics.written
assert not etl.input_queue
assert not etl.output_queue
assert not etl.retry_queue

etl.metrics, etl.dead_letters


(ETLMetrics(produced=7, transformed=3, written=3, retries=3, dead_letters=4, blocked_input=0, blocked_output=0, turns=5),
 [({'id': -3, 'name': 'invalid id'}, 'id must be a positive integer'),
  ({'name': 'missing id'}, 'missing id'),
  ({'id': 4, 'name': '   '}, 'name cannot be blank'),
  ({'id': 5, 'name': 'alan turing'},
   'retry exhausted: simulated temporary dependency failure')])

# Problem 15 — Additional solved exercises

The remaining examples add more reusable patterns:

1. batching coroutine;
2. debounce by logical ticks;
3. round-robin flattening;
4. O(1) moving average;
5. fault-isolated broadcast.


## 15.1 — Batching coroutine

Collect complete batches of size `n`. On close, optionally flush the final partial batch.


In [34]:
@coroutine
def batcher(
    size: int,
    target: Generator[None, tuple[T, ...], Any],
    *,
    flush_partial: bool = True,
) -> Generator[None, T, None]:
    if size <= 0:
        raise ValueError("size must be positive")

    batch: list[T] = []

    try:
        while True:
            item = yield
            batch.append(item)

            if len(batch) == size:
                target.send(tuple(batch))
                batch.clear()
    finally:
        try:
            if batch and flush_partial:
                target.send(tuple(batch))
        finally:
            target.close()


batches: list[tuple[int, ...]] = []
batch_pipeline = batcher(3, collect(batches), flush_partial=True)

for value in range(8):
    batch_pipeline.send(value)

batch_pipeline.close()

assert batches == [
    (0, 1, 2),
    (3, 4, 5),
    (6, 7),
]

batches


[(0, 1, 2), (3, 4, 5), (6, 7)]

## 15.2 — Debounce by logical ticks

Emit a value only after it remains the latest value for at least `delay` ticks.


In [35]:
def debounce_events(
    events: Iterable[tuple[int, T]],
    *,
    delay: int,
) -> list[tuple[int, T]]:
    if delay < 0:
        raise ValueError("delay cannot be negative")

    iterator = iter(events)

    try:
        pending_tick, pending_value = next(iterator)
    except StopIteration:
        return []

    output: list[tuple[int, T]] = []

    for tick, value in iterator:
        if tick < pending_tick:
            raise ValueError("events must be sorted by tick")

        if tick - pending_tick >= delay:
            output.append((pending_tick + delay, pending_value))

        pending_tick = tick
        pending_value = value

    output.append((pending_tick + delay, pending_value))
    return output


debounced = debounce_events(
    [(0, "A"), (1, "B"), (2, "C"), (7, "D"), (8, "E")],
    delay=3,
)

assert debounced == [(5, "C"), (11, "E")]
debounced


[(5, 'C'), (11, 'E')]

## 15.3 — Round-robin flattening

Take one item from each iterable in turn and remove exhausted iterators.


In [36]:
def round_robin(*iterables: Iterable[T]) -> Iterator[T]:
    active: deque[Iterator[T]] = deque(map(iter, iterables))

    while active:
        iterator = active.popleft()

        try:
            item = next(iterator)
        except StopIteration:
            continue
        else:
            yield item
            active.append(iterator)


flattened = list(
    round_robin(
        [1, 2, 3],
        ["A", "B"],
        [True, False, True, False],
    )
)

assert flattened == [
    1, "A", True,
    2, "B", False,
    3, True,
    False,
]

flattened


[1, 'A', True, 2, 'B', False, 3, True, False]

## 15.4 — O(1) moving average

Maintain a running sum so each update avoids recomputing `sum(window)`.


In [37]:
class MovingAverage:
    def __init__(self, window_size: int) -> None:
        if window_size <= 0:
            raise ValueError("window_size must be positive")

        self._window: deque[float] = deque(maxlen=window_size)
        self._sum = 0.0

    def add(self, value: float) -> float:
        value = float(value)

        if len(self._window) == self._window.maxlen:
            self._sum -= self._window[0]

        self._window.append(value)
        self._sum += value

        return self._sum / len(self._window)

    def snapshot(self) -> tuple[float, ...]:
        return tuple(self._window)


moving_average = MovingAverage(3)
averages = [moving_average.add(value) for value in [1, 2, 3, 4, 5]]

assert averages == [1.0, 1.5, 2.0, 3.0, 4.0]
assert moving_average.snapshot() == (3.0, 4.0, 5.0)

averages


[1.0, 1.5, 2.0, 3.0, 4.0]

## 15.5 — Fault-isolated broadcast

A failing branch must not prevent healthy branches from receiving later values.


In [38]:
@dataclass(frozen=True)
class BranchFailure:
    branch_index: int
    error_type: str
    message: str


@coroutine
def fault_isolated_broadcast(
    targets: Iterable[Generator[None, T, Any]],
    failures: list[BranchFailure],
) -> Generator[None, T, None]:
    active = list(enumerate(targets))

    try:
        while True:
            item = yield
            survivors: list[tuple[int, Generator[None, T, Any]]] = []

            for index, target in active:
                try:
                    target.send(item)
                except BaseException as error:
                    if isinstance(
                        error,
                        (GeneratorExit, KeyboardInterrupt, SystemExit),
                    ):
                        raise

                    failures.append(
                        BranchFailure(
                            branch_index=index,
                            error_type=type(error).__name__,
                            message=str(error),
                        )
                    )
                    target.close()
                else:
                    survivors.append((index, target))

            active = survivors
    finally:
        for _, target in active:
            target.close()


@coroutine
def fail_on(
    forbidden: int,
    output: list[int],
) -> Generator[None, int, None]:
    while True:
        value = yield

        if value == forbidden:
            raise ValueError(f"forbidden value: {value}")

        output.append(value)


In [39]:
healthy_output: list[int] = []
fragile_output: list[int] = []
branch_failures: list[BranchFailure] = []

fanout = fault_isolated_broadcast(
    [
        collect(healthy_output),
        fail_on(3, fragile_output),
    ],
    branch_failures,
)

for value in [1, 2, 3, 4, 5]:
    fanout.send(value)

fanout.close()

assert healthy_output == [1, 2, 3, 4, 5]
assert fragile_output == [1, 2]
assert len(branch_failures) == 1
assert branch_failures[0].branch_index == 1

healthy_output, fragile_output, branch_failures


([1, 2, 3, 4, 5],
 [1, 2],
 [BranchFailure(branch_index=1, error_type='ValueError', message='forbidden value: 3')])

# Further challenge problems

Try these without looking up a complete implementation:

1. Add priorities and priority aging to `TimedScheduler`.
2. Add task dependencies and a `WaitTask(task_id)` command.
3. Implement a bounded channel with separate sender and receiver objects.
4. Add a timeout command that injects an exception into a sleeping task.
5. Translate the producer-consumer simulation to `asyncio.Queue`.
6. Add exponential retry backoff to the ETL capstone.
7. Add graceful shutdown with a logical deadline.
8. Add structured event logs and per-task latency metrics.
9. Property-test `BoundedFIFO` with random operation sequences.
10. Compare cooperative scheduling, threads, and processes for CPU-bound work.


# Final best-practices checklist

- Yield frequently enough to preserve responsiveness.
- Make queue overflow explicit.
- Use `deque(maxlen=...)` only when automatic eviction is intentional.
- Keep priming, running, cancellation, and shutdown visible.
- Use `finally` for cleanup.
- Never swallow `GeneratorExit`.
- Isolate failures at task and pipeline boundaries.
- Test invariants, not only example outputs.
- Use logical time for deterministic scheduler tests.
- Benchmark fresh fixtures.
- Prefer `asyncio` for production asynchronous I/O.
